In [74]:
import pandas as pd
import kagglehub
import os
import glob

In [75]:
# 1. IMPORTAR A BASE DA IBM (Tabela Fato)
# ==========================================
# Faz o download direto do Kaggle como estava no seu código original
caminho = kagglehub.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")
arquivos_csv = glob.glob(os.path.join(caminho, "*.csv"))
df = pd.read_csv(arquivos_csv[0])

In [76]:
# 2. CRIAR A DIMENSÃO DE CARGOS IBM
# ==========================================
# Isola os cargos únicos e cria IDs para eles
cargos_unicos_ibm = sorted(df["JobRole"].dropna().unique())
df_dim_cargo_ibm = pd.DataFrame(cargos_unicos_ibm, columns=["JobRole"])
df_dim_cargo_ibm.insert(0, "id_cargo", range(1, len(df_dim_cargo_ibm) + 1))

In [77]:
# Substitui o texto do cargo pelo 'id_cargo' na tabela Fato principal
df = df.merge(df_dim_cargo_ibm, on="JobRole", how="left")
df = df.drop(columns=["JobRole"])

In [78]:
# 3. IMPORTAR AS TABELAS DE RANKING (Excel)
# ==========================================
# Certifique-se de que o ficheiro excel está na mesma pasta do seu VS Code
nome_arquivo = "db_a3_engenharia_de_dados.xlsx"
df_custo = pd.read_excel(nome_arquivo, sheet_name="ranking_custo_trabalho")
df_salario = pd.read_excel(nome_arquivo, sheet_name="ranking_salario")

print("Sucesso! As variáveis df, df_dim_cargo_ibm, df_custo e df_salario estão na memória.")

Sucesso! As variáveis df, df_dim_cargo_ibm, df_custo e df_salario estão na memória.


In [79]:
import pandas as pd
from sqlalchemy import create_engine, text

print("="*60)
print("ETAPA 1: CONEXÃO COM O BANCO DE DADOS (SQLITE)")
print("="*60)



ETAPA 1: CONEXÃO COM O BANCO DE DADOS (SQLITE)


In [80]:
# Cria o arquivo do banco de dados na mesma pasta do projeto
engine = create_engine('sqlite:///projeto_a3.db')
print("✅ Banco de dados 'projeto_a3.db' criado/conectado com sucesso na pasta local.\n")

print("="*60)

✅ Banco de dados 'projeto_a3.db' criado/conectado com sucesso na pasta local.



In [81]:
print("="*60)
print("ETAPA 2: CARGA DO STAR SCHEMA E TABELAS ADICIONAIS")
print("="*60)

# Carga das Tabelas e exibição da volumetria
df.to_sql('fato_ibm', con=engine, if_exists='replace', index=False)
print(f"✅ Tabela Fato 'fato_ibm' carregada! Total: {len(df)} linhas.")

ETAPA 2: CARGA DO STAR SCHEMA E TABELAS ADICIONAIS
✅ Tabela Fato 'fato_ibm' carregada! Total: 1470 linhas.


In [82]:
df_dim_cargo_ibm.to_sql('dim_cargo_ibm', con=engine, if_exists='replace', index=False)
print(f"✅ Tabela Dimensão 'dim_cargo_ibm' carregada! Total: {len(df_dim_cargo_ibm)} linhas.")

✅ Tabela Dimensão 'dim_cargo_ibm' carregada! Total: 9 linhas.


In [83]:
df_custo.to_sql('ranking_custo', con=engine, if_exists='replace', index=False)
print(f"✅ Tabela Adicional 'ranking_custo' carregada! Total: {len(df_custo)} linhas.")

✅ Tabela Adicional 'ranking_custo' carregada! Total: 30 linhas.


In [84]:
df_salario.to_sql('ranking_salario', con=engine, if_exists='replace', index=False)
print(f"✅ Tabela Adicional 'ranking_salario' carregada! Total: {len(df_salario)} linhas.\n")

✅ Tabela Adicional 'ranking_salario' carregada! Total: 30 linhas.



In [99]:
print("="*60)
print("ETAPA 3: APLICAÇÃO DA GOVERNANÇA DE DADOS (RLS)")
print("="*60)

ETAPA 3: APLICAÇÃO DA GOVERNANÇA DE DADOS (RLS)


In [101]:

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS controle_acesso"))
    conn.execute(text("""
        CREATE TABLE controle_acesso (
            usuario VARCHAR(50), 
            id_cargo INT
        )
    """))
    
    # Cadastrando múltiplos usuários associados aos IDs reais da tabela dim_cargo_ibm
    conn.execute(text("INSERT INTO controle_acesso VALUES ('GestorRH', 2)"))
    conn.execute(text("INSERT INTO controle_acesso VALUES ('CoordenadorLab', 3)"))
    conn.execute(text("INSERT INTO controle_acesso VALUES ('LiderPesquisa', 7)"))
    conn.execute(text("INSERT INTO controle_acesso VALUES ('GestorVendas', 8)"))
    conn.execute(text("INSERT INTO controle_acesso VALUES ('DiretorGeral', NULL)"))
    
    conn.execute(text("DROP VIEW IF EXISTS vw_fato_ibm_segura"))
    conn.execute(text("""
        CREATE VIEW vw_fato_ibm_segura AS
        SELECT f.*, ca.usuario 
        FROM fato_ibm f
        INNER JOIN controle_acesso ca 
          ON (ca.id_cargo = f.id_cargo OR ca.id_cargo IS NULL)
    """))
    conn.commit()

print("✅ Matriz de segurança atualizada com 5 usuários diferentes!\n")

print("="*60)
print("EXPORTAÇÃO OFICIAL PARA O POWER BI")
print("="*60)

# Puxa a View inteira (sem filtro WHERE) para trazer todos os usuários criados acima
query_completa = "SELECT * FROM vw_fato_ibm_segura"
df_completo_pbi = pd.read_sql(query_completa, engine)

# Exporta para o Excel
with pd.ExcelWriter('dados_para_powerbi.xlsx') as writer:
    df_completo_pbi.to_excel(writer, sheet_name='Fato_IBM_Segura', index=False)
    df_dim_cargo_ibm.to_excel(writer, sheet_name='Dim_Cargos', index=False)
    df_custo.to_excel(writer, sheet_name='Ranking_Custo', index=False)
    df_salario.to_excel(writer, sheet_name='Ranking_Salario', index=False)

print(f"✅ Arquivo 'dados_para_powerbi.xlsx' gerado!")
print(f"Usuários disponíveis para o seu filtro no Power BI: {df_completo_pbi['usuario'].unique()}")

✅ Matriz de segurança atualizada com 5 usuários diferentes!

EXPORTAÇÃO OFICIAL PARA O POWER BI
✅ Arquivo 'dados_para_powerbi.xlsx' gerado!
Usuários disponíveis para o seu filtro no Power BI: <StringArray>
['GestorVendas', 'DiretorGeral', 'LiderPesquisa', 'CoordenadorLab',
 'GestorRH']
Length: 5, dtype: str


In [102]:
print("="*60)
print("ETAPA 4: TESTE DE VALIDAÇÃO DO RLS PARA APRESENTAÇÃO")
print("="*60)

ETAPA 4: TESTE DE VALIDAÇÃO DO RLS PARA APRESENTAÇÃO


In [103]:
# Simula o login do Gestor
print("🔍 LOGIN: GESTOR DE VENDAS (Acesso Restrito)")
query_gestor = """
    SELECT EmployeeNumber, id_cargo, MonthlyIncome, usuario 
    FROM vw_fato_ibm_segura 
    WHERE usuario = 'GestorVendas'
"""
df_teste_gestor = pd.read_sql(query_gestor, engine)
display(df_teste_gestor.head(3))
print(f"⚠️ Atenção: O Gestor enxerga apenas {len(df_teste_gestor)} linhas de toda a base.\n")

🔍 LOGIN: GESTOR DE VENDAS (Acesso Restrito)


,EmployeeNumber,id_cargo,MonthlyIncome,usuario
0,1,8,5993,GestorVendas
1,35,8,6825,GestorVendas
2,52,8,5376,GestorVendas


⚠️ Atenção: O Gestor enxerga apenas 326 linhas de toda a base.



In [104]:
# Simula o login do Diretor
print("🔍 LOGIN: DIRETOR GERAL (Acesso Total)")
query_diretor = """
    SELECT EmployeeNumber, id_cargo, MonthlyIncome, usuario 
    FROM vw_fato_ibm_segura 
    WHERE usuario = 'DiretorGeral'
"""
df_teste_diretor = pd.read_sql(query_diretor, engine)
display(df_teste_diretor.head(3))
print(f"🔓 Liberado: O Diretor enxerga todas as {len(df_teste_diretor)} linhas da base.")

🔍 LOGIN: DIRETOR GERAL (Acesso Total)


,EmployeeNumber,id_cargo,MonthlyIncome,usuario
0,1,8,5993,DiretorGeral
1,2,7,5130,DiretorGeral
2,4,3,2090,DiretorGeral


🔓 Liberado: O Diretor enxerga todas as 1470 linhas da base.


In [105]:
import pandas as pd
import numpy as np # Necessário para cálculos matemáticos e aleatoriedade

print("A preparar a base completa para o Power BI...")

# 1. Puxar a View INTEIRA
query_completa = "SELECT * FROM vw_fato_ibm_segura"
df_completo_pbi = pd.read_sql(query_completa, engine)

# ==========================================
# 2. O EMBARALHAMENTO (SHUFFLE)
# ==========================================
# Pega os dados atuais da coluna e os devolve em ordem aleatória
df_completo_pbi['usuario'] = np.random.permutation(df_completo_pbi['usuario'].values)
print("🎲 Valores da coluna 'usuario' embaralhados com sucesso!")

# 3. Exportar para Excel
with pd.ExcelWriter('dados_para_powerbi.xlsx') as writer:
    df_completo_pbi.to_excel(writer, sheet_name='Fato_IBM_Segura', index=False)
    df_dim_cargo_ibm.to_excel(writer, sheet_name='Dim_Cargos', index=False)
    df_custo.to_excel(writer, sheet_name='Ranking_Custo', index=False)
    df_salario.to_excel(writer, sheet_name='Ranking_Salario', index=False)

print("✅ Arquivo Excel atualizado e pronto para o Power BI!")

A preparar a base completa para o Power BI...
🎲 Valores da coluna 'usuario' embaralhados com sucesso!
✅ Arquivo Excel atualizado e pronto para o Power BI!
